<a href="https://colab.research.google.com/github/aromanenko/DSCS/blob/main/DSCS_9_Replenishment_Automation_10SKU_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

HSE, DSCS. Analytics in Retail Chain, Fall 2025

Lesson #3: Inventory Optimization in Retail

Alexey Romanenko, alexromsput@gmail.com

In [5]:
import pandas as pd
import numpy as np

pd.options.plotting.backend = "plotly"

import warnings
warnings.filterwarnings("ignore")

In [2]:
!pip install ipynb

In [6]:
# loading functions
def read_by_mask(path, mask, **args):
    filenames = next(walk(path), (None, None, []))[2]  # [] if no file
    df = pd.DataFrame()
    for xlsx in filenames:
        if mask in xlsx:
            df = pd.concat([df, pd.read_excel(path+xlsx, **args)])
    return df

# import data (early precalculated) from yandex-disk

import requests
from urllib.parse import urlencode

def read_csv_from_yd(public_key, **args):
  '''public_key - ссылка с доступом на скачивание файла
  '''
  base_url = 'https://cloud-api.yandex.net/v1/disk/public/resources/download?'

  # Получаем загрузочную ссылку
  final_url = base_url + urlencode(dict(public_key=public_key))
  response = requests.get(final_url)
  download_url = response.json()['href']


  return pd.read_csv(download_url, **args)

def read_xlsx_from_yd(public_key, **args):
  '''public_key - ссылка с доступом на скачивание файла
  '''
  base_url = 'https://cloud-api.yandex.net/v1/disk/public/resources/download?'

  # Получаем загрузочную ссылку
  final_url = base_url + urlencode(dict(public_key=public_key))
  response = requests.get(final_url)
  download_url = response.json()['href']


  return pd.read_excel(download_url)

In [9]:
# from ipynb.fs.full.IPO_functions import(qualityWAMAXPE, get_density_forecast, get_percentile,reshape_barchart,
#                                         agg_density_forecast, purchase_sh)
from ipynb.fs.full.IPO_functions import(calculate_norms, build_density_forecast, read_by_mask, read_csv_from_yd, read_xlsx_from_yd)

In [10]:
# read data to order_upto_level

# sales and stock history
abt = read_xlsx_from_yd('https://disk.yandex.ru/i/DSuNUdonsjd_rQ')
abt['period_dt'] = pd.to_datetime(abt['period_dt'], format = '%d.%m.%Y')
abt.set_index('period_dt', inplace = True)

# forecasts from demand forecast system (upstream process)
d_forecast = read_xlsx_from_yd('https://disk.yandex.ru/i/2CNR9vVCZUV8gA') # , parse_dates = ['period_dt'], format = '%d.%m.%Y').set_index('period_dt')
d_forecast['period_dt'] = pd.to_datetime(d_forecast['period_dt'], format = '%d.%m.%Y')
d_forecast .set_index('period_dt', inplace = True)

# pipelines in Retail chain
d_pipeline = read_xlsx_from_yd('https://disk.yandex.ru/i/Erost2rqavvUHw')

# order_upto_level from experts (to compare with data-driven order_upto_level)
current_norms = read_xlsx_from_yd('https://disk.yandex.ru/i/9O4UTlXhjdJCjA')

# order_upto_level from ML system
upd_norms =  read_xlsx_from_yd('https://disk.yandex.ru/i/97eXfTsm1BN5IA')

# load cost price
price = read_xlsx_from_yd('https://disk.yandex.ru/i/PZHq3abUiGTz6A')

In [ ]:
# sales and stock data
abt.head()

,product_id,location_id,s_qty,s_amount,promo,stock,price_regular,price_promo
period_dt,,,,,,,,
2019-01-01,1000,2002,NaN,NaN,NaN,1.0,NaN,NaN
2019-01-02,1000,2002,NaN,NaN,NaN,1.0,NaN,NaN
2019-01-03,1000,2002,NaN,NaN,NaN,1.0,NaN,NaN
2019-01-04,1000,2002,NaN,NaN,NaN,1.0,NaN,NaN
2019-01-05,1000,2002,NaN,NaN,NaN,1.0,NaN,NaN


In [ ]:
# forecast data (constant got the whole forecasting period)
d_forecast

,product_id,location_id,forecast_qty
period_dt,,,
2022-06-01,599,2002,2
2022-06-02,599,2002,2
2022-06-03,599,2002,1
2022-06-04,599,2002,2
2022-06-05,599,2002,2
...,...,...,...
2022-06-26,494232,2002,0
2022-06-27,494232,2002,1
2022-06-28,494232,2002,0


In [ ]:
# prepare full list of pairs product-location
assort_matrix = abt.groupby(['product_id', 'location_id']).count()[['s_qty']].\
    merge(d_forecast.groupby(['product_id', 'location_id']).count()[['forecast_qty']],
             how = 'left', left_index = True , right_index = True).reset_index()

assort_matrix.head()

,product_id,location_id,s_qty,forecast_qty
0,33,2002,0,NaN
1,34,2002,3,NaN
2,1000,2002,1,NaN
3,1542,2002,31,NaN
4,1546,2002,39,NaN


In [ ]:
# number of sku-location pairs
assort_matrix.shape

(859, 4)

In [ ]:
# look at original stock and sales data
_id = list(abt.groupby(['product_id', 'location_id']).count().index)

time_lvl = 'D'
for i in _id[10:13]:
    _up = np.ceil(abt[(abt['product_id'] == i[0]) & (abt['location_id'] == i[1])]['s_qty']\
                  .resample(time_lvl).sum().max())*2

    fig = abt[(abt['product_id'] == i[0]) & (abt['location_id'] == i[1])][['s_qty','stock']]\
    .resample(time_lvl).sum().clip(lower = -5.0,upper = _up).plot(title = 'product = {0}, location = {1}'.format(i[0], i[1]))
    fig.update_layout(autosize=False,width=1000,height=400,).show()

# Service Level

In [ ]:
# read service level data
sl = read_xlsx_from_yd('https://disk.yandex.ru/i/AjPKazwqppeddw')
sl.head()

# unique values for each pair sku-location
SL = sl.groupby('product_id').min()[['SL']]

In [ ]:
SL.head()

,SL
product_id,
33,0.70
34,0.70
229,0.96
279,0.70
284,0.70


# Lead time and constraints

In [ ]:
# read lt data
lt =  read_xlsx_from_yd('https://disk.yandex.ru/i/uWDMkdq5PjhEZw').rename(columns={'Частота поставок':'PBR'})
# lt.head()
lt['BS_policy'] = 1
LT = lt.groupby('product_id').min()\
    [['LT', 'Квант', 'MerchMinimum', 'PBR','BS_policy']]

# if LT = 0 replace with 1 (verything that arrives during the day arrives in a day)
lt.loc[lt[lt['LT']==0].index, 'LT'] = 1

In [ ]:
LT

,LT,Квант,MerchMinimum,PBR,BS_policy
product_id,,,,,
33,1,1,1,3,1
34,1,1,1,3,1
229,1,30,1,3,1
279,1,1,0,3,1
284,1,5,2,3,1
...,...,...,...,...,...
534677,1,1,0,3,1
534678,1,1,0,3,1
534679,1,1,0,3,1


In [ ]:
# calculate default forecast
# this step could be skeeped if original forecast is perfect
window = 7*8
fcst_start_dt = max(abt.index.max()+pd.Timedelta(days=1), d_forecast.index.min())
fcst_end_dt = max(abt.index.max()+pd.Timedelta(days=30), d_forecast.index.max())


# default forecast - mean for previous 8 weeks
loc_forecast = abt.clip(lower=0).loc[str(abt.index.max() - pd.Timedelta(days=window)):]\
    .groupby(['product_id', 'location_id']).agg({'sum','count'})['s_qty']/window

# upgrade upstream-demand-forecast-system values by default forecast
upd_d_forecast = d_forecast.loc[str(fcst_start_dt):].reset_index().merge(loc_forecast[['sum']].clip(lower = 0.001).reset_index(),
                how = 'outer', left_on = ['product_id', 'location_id'],
                     right_on = ['product_id', 'location_id'])

# coalesce between original and default
upd_d_forecast.iloc[upd_d_forecast.period_dt.isnull(),upd_d_forecast.columns.get_loc('period_dt')] = fcst_start_dt

# compare updated forecast with original
upd_d_forecast = upd_d_forecast.set_index(['product_id', 'location_id','period_dt']).unstack(['product_id', 'location_id'])\
    .reindex(pd.date_range(fcst_start_dt, fcst_end_dt, freq='D')).bfill().ffill()\
    .stack(['product_id', 'location_id']).reset_index(level=[1,2]) #.rename(index = {'period_dt'})
upd_d_forecast['forecast_qty'] = upd_d_forecast['forecast_qty'].combine_first(upd_d_forecast['sum'])

In [ ]:
upd_d_forecast.tail()

,product_id,location_id,forecast_qty,sum
2022-06-30,529209,2002,0.035714,0.035714
2022-06-30,529210,2002,0.160714,0.160714
2022-06-30,529211,2002,0.107143,0.107143
2022-06-30,531335,2002,0.714286,0.714286
2022-06-30,531336,2002,1.553571,1.553571


# Calculating of Order_Upto_Level
Note: here and below order-upto-level is denoted as norma

In [ ]:
maps = {
        'в пути':'pipeline',
        'СЗ':'safety_stock',
        'норма':'norma_level',
        'чистая норма':'clear_norm_level',
        'прогноз':'forecast',
        'дата заказа':'order_date',
        'точка заказа':'reorder_level',
        'квант sl':'quant sl',
        'пред квант sl':'sl before quant',
        'квант норма':'quant norm',
        'квант':'quant',
        'мерчминимум':'merchendize minimum'
        }
norms.rename(columns = maps).columns

Index(['product_id', 'location_id', 'stock', 'order_date', 'forecast',
       'norma_level', 'reorder_level', 'quant sl', 'sl before quant',
       'clear_norm_level', 'quant norm', 'quant', 'merchendize minimum',
       'pipeline', 'safety_stock'],
      dtype='object')

In [ ]:
horizon = 60  # horizon in days

# cycle for each product-location
for num in _id[10:13]:
    fcst_density = build_density_forecast(abt, pair=num, horizon = horizon, d_forecast = upd_d_forecast, fcst_id = 25)
    norms = calculate_norms(num, abt, fcst_density, lt, sl)
#     norms.to_csv(path+'norms_'+str(num[0])+'_'+str(num[1])+'.csv')
    norms.rename(columns = maps, inplace = True)
    fig = norms[['stock', 'pipeline', 'safety_stock', 'norma_level', 'clear_norm_level','forecast', 'order_date']]\
        .plot(title = 'product = {0} location = {1}<br>service level = {2}, frequency of supply = {6}, lead time = {3}, quant = {4}, Merchendise minimum = {5}<br>, expert norm={7}, current norm={8}'\
              .format(num[0], num[1],
                      sl[(sl['product_id'] == num[0])&(sl['location_id'] == num[1])].SL.values[0],
                      lt[(lt['product_id'] == num[0])&(lt['location_id'] == num[1])].LT.values[0],
                      lt[(lt['product_id'] == num[0])&(lt['location_id'] == num[1])]['Квант'].values[0],
                      lt[(lt['product_id'] == num[0])&(lt['location_id'] == num[1])].MerchMinimum.values[0],
                      lt[(lt['product_id'] == num[0])&(lt['location_id'] == num[1])].PBR.values[0],
                      np.nan, # current_norms[current_norms['код'] == num[0]]['норма'].values[0],
                      np.nan #  upd_norms[upd_norms['SKU'] == num[0]]['С_З'].values[0]
                      ))\
        .update_layout(height=350, width=950)

    fig.show()

    # fig.write_html(path+'/html/product_'+str(num[0])+'.html',
    #             full_html=False,
    #             include_plotlyjs='cdn')

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning:

Optimization failed to converge. Check mle_retvals.



# Analytical normas vs expert-normas

In [ ]:
# Analysis
groupby_columns = ['product_id','location_id']

# precalculated (order_upto_level)
full_norms = read_csv_from_yd('https://disk.yandex.ru/d/P6OcNjrZoUA43w', sep = ',', parse_dates = ['index'], dayfirst = True)

# set dates column as index
full_norms = full_norms.rename(columns = {'index':'period_dt'}).rename(columns = maps).set_index('period_dt')


# full_norms
period_end = '2022-06-30'
start_stock = full_norms.loc['2022-06-28'].groupby(groupby_columns).sum()[['stock']]

join_norms = full_norms.sort_index().loc[:period_end].\
    groupby(groupby_columns).\
    agg({'norma_level':['min', 'max', 'mean'], 'merchendize minimum':['min'],'quant': ['min'],'clear_norm_level':['min']})

# transform pivot
join_norms.columns = [' '.join(col).strip() for col in join_norms.columns.values]


# add some statistics to compare experts vs new norms
join_norms = join_norms.\
    merge(price.set_index(groupby_columns), how='left', left_index = True, right_index = True) .\
    merge(upd_d_forecast.groupby(['product_id', 'location_id']).mean()[['forecast_qty']],
             how = 'left', left_index = True , right_index = True).fillna(-1).\
    merge(start_stock, how='left', left_index = True, right_index = True).reset_index().\
    merge(upd_norms, how = 'left', left_on='product_id', right_on = 'SKU').\
    merge(current_norms, how = 'left', left_on='product_id', right_on = 'код')


join_norms.head()
# join_norms.to_csv('сравнение норм.csv', sep=';', index=False)

,product_id,location_id,norma_level min,norma_level max,norma_level mean,merchendize minimum min,quant min,clear_norm_level min,ССЦ,forecast_qty,stock,SKU,С_З,код,норма
0,33.0,2002.0,1.0,1.0,1.0,1,1,1.0,3582.00,-1.0,0.0,NaN,NaN,33,0
1,34.0,2002.0,1.0,1.0,1.0,1,1,1.0,4436.38,-1.0,0.0,NaN,NaN,34,0
2,229.0,2002.0,25.0,25.0,25.0,1,30,25.0,163.47,-1.0,1.0,229.0,20.0,229,20
3,279.0,2002.0,1.0,1.0,1.0,0,1,1.0,0.00,-1.0,1.0,NaN,NaN,279,0
4,284.0,2002.0,2.0,2.0,2.0,2,5,1.0,68.62,-1.0,8.0,284.0,5.0,284,5


In [ ]:
join_norms.head()

,product_id,location_id,new_norma,norma_level max,norma_level mean,merchendize minimum,quant,clear_norm_level min,ССЦ,forecast_qty,stock,SKU,С_З,код,норма
0,33.0,2002.0,1.0,1.0,1.0,1,1,1.0,3582.00,-1.0,0.0,NaN,NaN,33,0
1,34.0,2002.0,1.0,1.0,1.0,1,1,1.0,4436.38,-1.0,0.0,NaN,NaN,34,0
2,229.0,2002.0,25.0,25.0,25.0,1,30,25.0,163.47,-1.0,1.0,229.0,20.0,229,20
3,279.0,2002.0,1.0,1.0,1.0,0,1,1.0,0.00,-1.0,1.0,NaN,NaN,279,0
4,284.0,2002.0,2.0,2.0,2.0,2,5,1.0,68.62,-1.0,8.0,284.0,5.0,284,5


In [ ]:
join_norms.columns

Index(['product_id', 'location_id', 'new_norma', 'norma_level max',
       'norma_level mean', 'merchendize minimum', 'quant',
       'clear_norm_level min', 'cost_price', 'forecast_qty', 'stock', 'SKU',
       'safety_stock', 'код', 'norm_level'],
      dtype='object')

In [ ]:
join_norms.rename(columns = {'С_З':'safety_stock', 'норма':'norm_level', 'ССЦ':'cost_price'}, inplace = True)

In [ ]:
# one-by-one analysis of norms
join_norms.rename(columns={'merchendize minimum min':'merchendize minimum', 'norma_level min':'new_norma',
                            'clear_norm_level min':'clear_norm_level', 'quant min':'quant'},
                  inplace = True) # join original norms and quants


join_norms['relation'] = np.floor(join_norms['safety_stock']/join_norms['new_norma']).clip(upper=2) # rough norma_old/norma_new
join_norms['is_merch'] = 1*(join_norms['new_norma']-join_norms['merchendize minimum'])/join_norms['new_norma']<0.1 # new norma significantly lower
join_norms['rough_norm'] = np.ceil(join_norms['safety_stock'].clip(upper = 3)/1)*1 #
join_norms['forecast_flg'] = 1*((join_norms['forecast_qty']>=0) & (join_norms['cost_price']>0)) # forecast_flag calculation (only products with forecast)
join_norms['time_to_norm'] = 1.0*abs(join_norms['stock'] - join_norms['new_norma']) # dif stock and new-norma
join_norms['detailed_relation'] = (join_norms['safety_stock']/join_norms['new_norma']) # norma_old/norma_new
join_norms['norm_change'] = (join_norms['safety_stock']/join_norms['norm_level']) # norma_old to new final norma

wm = lambda x: np.dot(x, join_norms.loc[x.index, "cost_price"])/np.sum(join_norms.loc[x.index, "cost_price"]) # normalized weighted relation
w = lambda x: np.sum(join_norms.loc[x.index, "cost_price"]) # weighted relation

turn = lambda x: np.dot(x+join_norms.loc[x.index, "quant"]/2, join_norms.loc[x.index, "cost_price"])\
  /np.dot(join_norms.loc[x.index, "forecast_qty"], join_norms.loc[x.index, "cost_price"]) # turnover

by_columns = ['rough_norm', 'forecast_flg', 'relation']

estim = join_norms\
    .groupby(by_columns)\
    .agg(pw_old=("norm_level", turn),
#          price_weighted_forecast=("forecast_qty", turn),
         pw_new=("new_norma", turn),
         pw_fnew=("clear_norm_level", turn)
         ,  weight = ('new_norma', w)
         , stabilization_time = ('time_to_norm',wm)
        ).clip(lower=0)

# estim.to_csv(path + 'results.csv', sep = ';', decimal = ',')
estim.reset_index(inplace = True)

estim[estim['forecast_flg']==1].set_index(by_columns).fillna(0)

# join_norms.groupby(['rough_norm', 'forecast_flg', 'is_merch', 'relation']).count()[['код']]\
#     .unstack(['rough_norm', 'forecast_flg']).fillna(0)

pw_old      pw_new     pw_fnew  \
rough_norm forecast_flg relation                                       
1.0        1             0.0       48.077118  316.568069  101.398368   
                         1.0      126.781564   93.985809   93.985809   
2.0        1             0.0       33.078661   72.976870   26.597941   
                         1.0       45.455128   38.171909   30.872059   
                         2.0       66.434882   30.925919   30.925919   
3.0        1            -383.0     26.364666    0.866365    0.866365   
                        -260.0    410.810811    5.135135    5.135135   
                        -29.0     135.491803   20.573770   20.573770   
                         0.0       14.672798   22.252222   15.759750   
                         1.0       29.413268   20.206210   19.094686   
                         2.0       40.609015   17.948456   17.526871   

                                     weight  stabilization_time  
rough_norm forecast_flg relation                                 
1.0        1             0.0       14162.54            8.421626  
                         1.0       30210.08            1.787048  
2.0        1             0.0       12366.60            7.141594  
                         1.0       23349.18            0.770246  
                         2.0       35188.89            3.630156  
3.0        1            -383.0        83.92          118.000000  
                        -260.0       273.11          340.000000  
                        -29.0         56.53          150.000000  
                         0.0      176209.95           12.943072  
                         1.0      113779.65           11.715069  
                         2.0       97875.73            6.380305

#  Business effect estimation

In [ ]:
# wm = lambda x: np.dot(x, join_norms.loc[x.index, "ССЦ"])/np.sum(join_norms.loc[x.index, "ССЦ"])

p_w = lambda x: np.dot(x, estim.loc[x.index, "weight"])/np.sum(estim.loc[x.index, "weight"])


# оборачиваемость по цене
estim['upd_turn'] = estim['pw_old']

# только те пары, где уверенны в принятии решения в пользу новой модели (не сильное отклонение от старой)
condition = (estim['forecast_flg']==1)&(estim['relation']>0)

estim.iloc[condition, estim.columns.get_loc('upd_turn')] = estim.loc[condition,'pw_new']

estim.reset_index()\
    .groupby('forecast_flg')\
    .agg(turnover_old=("pw_old", p_w),
        turnover_new=("upd_turn", p_w))



,turnover_old,turnover_new
forecast_flg,,
0,0.000000,0.000000
1,36.436819,25.165316


In [ ]:
# estimate
print('Daily stock cost of ~1% assortment {:.2f} mln rub'.format(estim.loc[condition, 'weight'].sum()/10**6))

# estimation of ML approach benefit to the company
print('Rough implementation value {:.2f} mln rub/per day'.format(100*(36-25)/36*estim.loc[condition, 'weight'].sum()/10**6))

Daily stock cost of ~1% assortment 0.30 mln rub
Rough implementation value 9.18 mln rub
